Connect Drive (so the file survives)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/NoShows"
DATA = BASE + "/data"

Mounted at /content/drive


Remove the hidden line breaks from the column names

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith(".xlsx"):
            print(os.path.join(root, file))

/content/drive/MyDrive/somthing's sia /FCT VENTES 2TR22.xlsx
/content/drive/MyDrive/somthing's sia /FCT VENTES 3TR22.xlsx
/content/drive/MyDrive/somthing's sia /Proposition planning SYPEX_AWB V1.1.xlsx
/content/drive/MyDrive/Noshows/data/Dataset_MedicalPM(fichier_nettoye).xlsx


In [ ]:
print(os.listdir("/content/drive/MyDrive/Noshows"))

['data', 'models', 'notebooks', 'reports']


Cell 2. Remove the hidden line breaks from the column names



In [ ]:
print(os.listdir("/content/drive/MyDrive/Noshows/data"))

['Dataset_MedicalPM(fichier_nettoye).xlsx']


In [ ]:
import pandas as pd

BASE = "/content/drive/MyDrive/Noshows"
DATA = BASE + "/data"

df = pd.read_excel(DATA + "/Dataset_MedicalPM(fichier_nettoye).xlsx")

print(df.shape)

(110527, 13)


In [ ]:
print(df.columns.tolist())

['ID patient', 'ID de rendez-vous', 'Genre', 'Spécialité médicale', 'Jour programmé', 'Jour de rendez-vous\n\n', 'Age', 'Quartier', 'Bourse', 'SMS_reçu', 'Non-présentation\n', 'Radio dentaire réalisée', 'Nom du dentiste']


remove duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print("Duplicates removed:", before - len(df))

Duplicates removed: 0


In [ ]:
print("Ages above 100:", (df["Age"] > 100).sum())

before = len(df)
df = df[df["Age"] <= 100]
print("Rows removed:", before - len(df))
print(df["Age"].min(), df["Age"].max())

Ages above 100: 0
Rows removed: 0
0 100


Problem de age -1


In [ ]:
age_median = df[df["Age"] >= 0]["Age"].median()

df.loc[df["Age"] < 0, "Age"] = age_median

In [ ]:
bad = df["Jour de rendez-vous"].dt.normalize() < df["Jour programmé"].dt.normalize()
print("Appointments before scheduling:", bad.sum())

df = df[~bad]
print("Rows left:", len(df))

KeyError: 'Jour de rendez-vous'

In [ ]:
print(df.columns.tolist())

['ID patient', 'ID de rendez-vous', 'Genre', 'Spécialité médicale', 'Jour programmé', 'Jour de rendez-vous\n\n', 'Age', 'Quartier', 'Bourse', 'SMS_reçu', 'Non-présentation\n', 'Radio dentaire réalisée', 'Nom du dentiste']


nettoyer le nom de column

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
print(df.columns.tolist())

['ID patient', 'ID de rendez-vous', 'Genre', 'Spécialité médicale', 'Jour programmé', 'Jour de rendez-vous', 'Age', 'Quartier', 'Bourse', 'SMS_reçu', 'Non-présentation', 'Radio dentaire réalisée', 'Nom du dentiste']


In [ ]:
df["Jour programmé"] = pd.to_datetime(df["Jour programmé"])
df["Jour de rendez-vous"] = pd.to_datetime(df["Jour de rendez-vous"])

In [ ]:
bad = df["Jour de rendez-vous"].dt.normalize() < df["Jour programmé"].dt.normalize()
print("Appointments before scheduling:", bad.sum())

df = df[~bad]
print("Rows left:", len(df))

Appointments before scheduling: 5
Rows left: 110515


Task 3: check the dates

In [ ]:
print(df[["Jour programmé", "Jour de rendez-vous"]].dtypes)

Jour programmé         datetime64[ns]
Jour de rendez-vous    datetime64[ns]
dtype: object


Task 4: fix the Age type and save

In [ ]:
df["Age"] = df["Age"].astype(int)

df.to_csv(DATA + "/data_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (110515, 13)


Check that the saved file is correct

In [ ]:
check = pd.read_csv(DATA + "/data_clean.csv")
print(check.shape)
print(check.columns.tolist())
print(check["Age"].min(), check["Age"].max())
print(check.isnull().sum().sum())

(110515, 13)
['ID patient', 'ID de rendez-vous', 'Genre', 'Spécialité médicale', 'Jour programmé', 'Jour de rendez-vous', 'Age', 'Quartier', 'Bourse', 'SMS_reçu', 'Non-présentation', 'Radio dentaire réalisée', 'Nom du dentiste']
0 100
0
